<a href="https://colab.research.google.com/github/maanaav15369/AI/blob/main/pytorch_rnn_based_qa_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [2]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [3]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [4]:
# vocab
vocab = {'<UNK>':0}

In [5]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)


In [6]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [7]:
len(vocab)

327

In [8]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [22]:
text_to_indices("Who is Maanaav Trivedi", vocab)

[10, 2, 34, 35]

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

In [11]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [12]:
dataset = QADataset(df, vocab)

In [13]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [14]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[45, 89, 90, 91, 92, 42, 93]]) tensor([94])
tensor([[ 10,   2,  65,  66,   3, 286,   5, 287]]) tensor([288])
tensor([[  1,   2,   3,   4,   5, 209]]) tensor([210])
tensor([[  1,   2,   3, 237,   5, 238]]) tensor([135])
tensor([[ 45, 203,   2,  14, 204, 205, 206, 207]]) tensor([208])
tensor([[  1,   2,   3,   4,   5, 289]]) tensor([290])
tensor([[10, 58,  3, 59,  5, 60]]) tensor([61])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]]) tensor([28])
tensor([[  1,   2,   3, 145, 120,  86,   3, 280, 281]]) tensor([124])
tensor([[ 45,  89,  90, 244, 245,  19,  42, 246]]) tensor([247])
tensor([[ 1,  2,  3,  4,  5, 76]]) tensor([77])
tensor([[ 45, 170,   2,   3,  17, 171, 172]]) tensor([173])
tensor([[  1,  90, 232, 233, 234, 235]]) tensor([236])
tensor([[ 45, 104,   2,   3,  17]]) tensor([105])
tensor([[ 45, 258,   2, 259,  86, 260, 261]]) tensor([262])
tensor([[ 45,  18,   2,   3, 284,  12,   3, 285]]) tensor([208])
tensor([[ 45, 141,   2,  65,  42,   3, 325, 326]]) tensor([6])
tensor([[

In [15]:
import torch.nn as nn

In [16]:
# understanding applied Here...
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [17]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [18]:
learning_rate = 0.001
epochs = 20

In [19]:
model = SimpleRNN(len(vocab))

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [23]:
import torch.nn as nn

# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    # Ensure target is a single token for CrossEntropyLoss
    loss = criterion(output, answer[0][0].unsqueeze(0))

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 477.616527
Epoch: 2, Loss: 402.462944
Epoch: 3, Loss: 334.836679
Epoch: 4, Loss: 281.649092
Epoch: 5, Loss: 232.056821
Epoch: 6, Loss: 187.387256
Epoch: 7, Loss: 147.702164
Epoch: 8, Loss: 114.319915
Epoch: 9, Loss: 87.959401
Epoch: 10, Loss: 67.622118
Epoch: 11, Loss: 52.511784
Epoch: 12, Loss: 41.471770
Epoch: 13, Loss: 33.183538
Epoch: 14, Loss: 27.431636
Epoch: 15, Loss: 22.713163
Epoch: 16, Loss: 18.992064
Epoch: 17, Loss: 16.003728
Epoch: 18, Loss: 13.674480
Epoch: 19, Loss: 11.920474
Epoch: 20, Loss: 10.288092


In [24]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [27]:
predict(model, "What is the name of the person who wote this project?")

maanaav


In [31]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'name': 29,
 'person': 30,
 'wote': 31,
 'this': 32,
 'project': 33,
 'maanaav': 34,
 'trivedi': 35,
 'square': 36,
 'root': 37,
 '64': 38,
 '8': 39,
 'chemical': 40,
 'symbol': 41,
 'for': 42,
 'gold': 43,
 'au': 44,
 'which': 45,
 'year': 46,
 'did': 47,
 'world': 48,
 'war': 49,
 'ii': 50,
 'end': 51,
 '1945': 52,
 'longest': 53,
 'river': 54,
 'nile': 55,
 'japan': 56,
 'tokyo': 57,
 'developed': 58,
 'theory': 59,
 'relativity': 60,
 'albert-einstein': 61,
 'freezing': 62,
 'fahrenheit': 63,
 '32': 64,
 'known': 65,
 'as': 66,
 'red': 67,
 'mars': 68,
 'author': 69,
 '1984': 70,
 'george-or

In [32]:
list(vocab.keys())[35]

'trivedi'

In [29]:
# extra for understanding
#solution
x =nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True) #[1,1,64]
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e =z(d.squeeze(0)) # removing extra dimension using unsqueeze

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [30]:
# problem
x =nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e =z(d)

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 6, 64])
shape of e: torch.Size([1, 6, 324])
